# MTKD4SER: Multi-Teacher Knowledge Distillation for Speech Emotion Recognition

This notebook provides a complete pipeline to train and evaluate the **MTKD4SER** model on **Google Colab (Free T4 GPU / 15GB VRAM)**.

### Key Highlights:
- **Language-Aware Dynamic Routing**: Oracle Language routing, Uniform, Cosine similarity, and Confidence routing.
- **VRAM Optimized**: Dynamic dual-teacher setup (English + French) avoiding Finnish teacher duplication, safely running on T4 GPU.
- **Hugging Face Hub Streaming & Arrow Support**: Direct loading from Hugging Face Hub (zero-copy Arrow tables) without Google Drive upload bottlenecks.
- **Speaker-Independent Splits**: 0% speaker leakage across Train, Dev, and Test sets.

## 1. Verify GPU Hardware
Colab provides Nvidia T4 (15 GB VRAM). Ensure CUDA is available.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive (Optional)
Mount Google Drive if you wish to persist trained checkpoints (`/content/checkpoints`) across Colab sessions. If using Hugging Face Hub for datasets and Weights & Biases for artifact tracking, mounting Drive is optional.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone Repository & Install Dependencies
Replace the repository URL below with your GitHub repository.

In [ ]:
# Clone the repository (or upload project folder)
!git clone https://github.com/nhatientri/mtkd4ser_phase1.git
%cd mtkd4ser_phase1

# Install project dependencies
!pip install -q transformers datasets torchaudio soundfile librosa scikit-learn tqdm tensorboard accelerate pyarrow wandb huggingface_hub


### Hugging Face Authentication & Hub Setup (Required for Private Repositories)
Setting your `HF_TOKEN` authenticates access to your Hugging Face dataset repository (e.g. `nhatientri/mtkd4ser-phase1`) and avoids rate limits.
You can store `HF_TOKEN` in Colab Secrets (the 🔑 icon on the left sidebar) or log in directly via `huggingface_hub.login()`.

In [ ]:
import os
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        login(token=hf_token)
        print("✓ Authenticated to Hugging Face via Colab Secrets.")
    else:
        print("ℹ No HF_TOKEN detected in Colab Secrets.")
        print("  If using a private Hugging Face dataset, log in interactively or set your token below.")
        # login()
except Exception as e:
    print(f"ℹ Hugging Face authentication note: {e}")
    # login(token="hf_...")


### Weights & Biases (W&B) Authentication (Recommended)
Logging to [Weights & Biases](https://wandb.ai) gives you live remote loss curves, teacher routing weight distributions, confusion matrices, and automatic cloud model artifact versioning.

Store your `WANDB_API_KEY` in Colab Secrets (the 🔑 icon on the left sidebar) or log in interactively.

In [ ]:
import os
try:
    from google.colab import userdata
    wandb_key = userdata.get("WANDB_API_KEY")
    if wandb_key:
        import wandb
        wandb.login(key=wandb_key)
        print("✓ W&B successfully authenticated from Colab Secrets.")
    else:
        print("ℹ No WANDB_API_KEY detected in Colab Secrets. You can log in interactively:")
        import wandb
        wandb.login()
except Exception as e:
    print(f"ℹ Interactive login fallback: {e}")
    # import wandb
    # wandb.login()


## 4. Dataset Preparation

### Option A: Hugging Face Hub (Recommended - Zero Drive Storage Required)
Load pre-processed datasets directly from your Hugging Face Hub repository (`ngunghoc/mtkd4ser-phase1`).

> **Why your Hugging Face repository looks like this:**
> On Hugging Face Hub, multi-subset datasets store each configuration as a folder at the root of the repository (`cafe/`, `combined/`, `iemocap_session1/`, etc.). They are **not** supposed to be under `data/hf_datasets/` on the Hub.
> When you call `load_dataset('ngunghoc/mtkd4ser-phase1', 'combined')`, Hugging Face automatically reads the `combined/` configuration folder.
>
> ⚠️ **Private Repository Authentication:** Because `ngunghoc/mtkd4ser-phase1` is private, make sure you ran the Hugging Face Authentication cell in Step 3 above with your `HF_TOKEN`.

### Option B: Extract Pre-packaged Archive from Google Drive (Alternative / Offline)
If you uploaded `hf_data.zip` to your Google Drive, you can extract it locally into `./data/hf_datasets/`.

In [ ]:
# ============================================================================
# OPTION A: Set Hugging Face Dataset Repository (Recommended)
# ============================================================================
HF_DATASET_REPO = "ngunghoc/mtkd4ser-phase1"
os.environ["HF_DATASET_REPO"] = HF_DATASET_REPO
print(f"✓ Active Hugging Face Dataset Repository: {HF_DATASET_REPO}")


In [ ]:
# ============================================================================
# OPTION B: Extract Pre-packaged Archive from Google Drive (Alternative)
# ============================================================================
# Uncomment if you prefer extracting local hf_data.zip from Drive instead of HF Hub:
# !unzip -q /content/drive/MyDrive/hf_data.zip -d ./

import os
if os.path.exists("data/hf_datasets"):
    print("✓ Local Hugging Face Arrow datasets found:", os.listdir("data/hf_datasets"))
else:
    print(f"ℹ No local data/hf_datasets found; training will stream directly from Hugging Face: {os.environ.get('HF_DATASET_REPO')}")


### Quick Dataset Verification
Test that the dataset loads properly from Hugging Face Hub (or local cache):

In [ ]:
import os
from data import build_multilingual, iemocap, cafe

hf_repo = os.environ.get("HF_DATASET_REPO", "ngunghoc/mtkd4ser-phase1")
print(f"Testing dataset loading (Repo: {hf_repo})...")

label2id, id2label, ds = build_multilingual(session_en=1, session_fr=1, hf_dataset_repo=hf_repo)
print("\n✓ Multilingual Dataset structure:")
print(ds)
print("Emotion labels:", label2id)
print("Train split features:", ds['train'].features)
print(f"Sample 0 emotion: {id2label[ds['train'][0]['label']]}, lang_id: {ds['train'][0]['language_id']}")


## 5. Train Monolingual Teacher Models (Optional if Checkpoints on W&B)
Before running MTKD, you need fine-tuned (FT) monolingual teachers for English and French.

> **Tip – Skip if already trained:**
> If you already trained the teacher models and logged them to Weights & Biases (W&B Artifacts), you can **skip Step 5 entirely**! You do not need `.pth` files stored locally or on Google Drive. In Step 6, the pipeline will automatically resolve and download your checkpoints directly from Weights & Biases.


In [ ]:
# 1. English Monolingual Teacher (IEMOCAP Session 1 Fold)
# Note: --RESUME_CKPT allows resuming from a remote W&B artifact (or local .pth file)
!python main.py --PARADIGM FT --LANGUAGE EN --LINGUALITY Monolingual \
    --SESSION 1 --N_EPOCHS 12 --BATCH_SIZE 8 --LEARNING_RATE 3e-5 \
    --TRAINING 1 --SAVE_DIR /content/checkpoints \
    --HF_DATASET_REPO "$HF_DATASET_REPO" \
    --RESUME_CKPT "mtkd4ser-phase1/FT_Monolingual_EN_S1-best:latest" \
    --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1


In [ ]:
# 2. French Monolingual Teacher (CaFE Fold 1)
# Note: --RESUME_CKPT allows resuming from a remote W&B artifact (or local .pth file)
!python main.py --PARADIGM FT --LANGUAGE FR --LINGUALITY Monolingual \
    --SESSION 1 --N_EPOCHS 12 --BATCH_SIZE 8 --LEARNING_RATE 3e-5 \
    --TRAINING 1 --SAVE_DIR /content/checkpoints \
    --HF_DATASET_REPO "$HF_DATASET_REPO" \
    --RESUME_CKPT "mtkd4ser-phase1/FT_Monolingual_FR_S1-best:latest" \
    --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1


## 6. Multi-Teacher Knowledge Distillation (MTKD)
Train student model using both English and French teachers.

### Recommended: Oracle Language-Aware Routing
Uses sample `language_id` (0=EN, 2=FR) to dynamically route loss to the native language teacher.

### Remote Checkpoint Resolution via Weights & Biases (W&B)
If your teacher checkpoints are stored on Weights & Biases rather than locally:
- Specify the W&B Artifact URI (e.g. `FT_Monolingual_EN_S1-best:best` or `entity/project/name:alias`) in `TEACHER_EN_CKPT` and `TEACHER_FR_CKPT`.
- `main.py` will automatically resolve and download the model files directly into `/content/checkpoints/` and register them as input artifacts in your W&B lineage graph.


In [ ]:
# ============================================================================
# Configure Teacher Checkpoints from Weights & Biases (or Local Paths)
# ============================================================================
# If stored on W&B, use the artifact format:
#   'artifact_name:alias' or '<project>/<artifact_name>:<alias>' or '<entity>/<project>/<artifact_name>:<alias>'
# Default names logged during Monolingual FT:
#   EN: 'mtkd4ser-phase1/FT_Monolingual_EN_S1-best:latest' (or :best)
#   FR: 'mtkd4ser-phase1/FT_Monolingual_FR_S1-best:latest' (or :best)
import os

WANDB_PROJECT = os.environ.get('WANDB_PROJECT', 'mtkd4ser-phase1')
WANDB_ENTITY = os.environ.get('WANDB_ENTITY', '')  # Set your W&B entity/username if needed

# Teacher Checkpoints: remote W&B artifacts or local paths
TEACHER_EN_CKPT = 'mtkd4ser-phase1/FT_Monolingual_EN_S1-best:latest'
TEACHER_FR_CKPT = 'mtkd4ser-phase1/FT_Monolingual_FR_S1-best:latest'

os.environ['TEACHER_EN_CKPT'] = TEACHER_EN_CKPT
os.environ['TEACHER_FR_CKPT'] = TEACHER_FR_CKPT
if WANDB_ENTITY:
    os.environ['WANDB_ENTITY'] = WANDB_ENTITY

print(f'✓ English Teacher Checkpoint: {TEACHER_EN_CKPT}')
print(f'✓ French Teacher Checkpoint:  {TEACHER_FR_CKPT}')


In [ ]:
# ============================================================================
# (Optional) Verify & Pre-download Teacher Checkpoints from W&B
# ============================================================================
# Run this cell to ensure your W&B teacher artifacts are accessible and download
# them to /content/checkpoints ahead of MTKD training:
from pipeline import resolve_checkpoint

print('Resolving English teacher checkpoint from W&B...')
resolved_en = resolve_checkpoint(TEACHER_EN_CKPT, save_dir='/content/checkpoints',
                                 default_project=WANDB_PROJECT, default_entity=WANDB_ENTITY or None)
print('Resolving French teacher checkpoint from W&B...')
resolved_fr = resolve_checkpoint(TEACHER_FR_CKPT, save_dir='/content/checkpoints',
                                 default_project=WANDB_PROJECT, default_entity=WANDB_ENTITY or None)

print(f'\n✓ English teacher ready at: {resolved_en}')
print(f'✓ French teacher ready at:  {resolved_fr}')


In [ ]:
!python main.py --PARADIGM MTKD --LANGUAGE EN --LINGUALITY Multilingual \
    --SESSION 1 --N_EPOCHS 12 --BATCH_SIZE 6 --LEARNING_RATE 3e-5 \
    --TRAINING 1 --ROUTING language \
    --TEACHER_EN_CKPT "$TEACHER_EN_CKPT" \
    --TEACHER_FR_CKPT "$TEACHER_FR_CKPT" \
    --INCLUDE_FI 0 --SAVE_DIR /content/checkpoints \
    --HF_DATASET_REPO "$HF_DATASET_REPO" \
    --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1


### Routing Ablations
Compare with baseline routing strategies:

In [ ]:
# Uniform baseline (50/50 teacher weights)
# !python main.py --PARADIGM MTKD --LANGUAGE EN --LINGUALITY Multilingual \
#     --SESSION 1 --N_EPOCHS 12 --BATCH_SIZE 6 --LEARNING_RATE 3e-5 \
#     --TRAINING 1 --ROUTING uniform \
#     --TEACHER_EN_CKPT "$TEACHER_EN_CKPT" \
#     --TEACHER_FR_CKPT "$TEACHER_FR_CKPT" \
#     --INCLUDE_FI 0 --SAVE_DIR /content/checkpoints \
#     --HF_DATASET_REPO "$HF_DATASET_REPO"

# Cosine similarity routing (Original paper approach)
# !python main.py --PARADIGM MTKD --LANGUAGE EN --LINGUALITY Multilingual \
#     --SESSION 1 --N_EPOCHS 12 --BATCH_SIZE 6 --LEARNING_RATE 3e-5 \
#     --TRAINING 1 --ROUTING cosine \
#     --TEACHER_EN_CKPT "$TEACHER_EN_CKPT" \
#     --TEACHER_FR_CKPT "$TEACHER_FR_CKPT" \
#     --INCLUDE_FI 0 --SAVE_DIR /content/checkpoints \
#     --HF_DATASET_REPO "$HF_DATASET_REPO"

# Confidence routing (Max softmax probability)
# !python main.py --PARADIGM MTKD --LANGUAGE EN --LINGUALITY Multilingual \
#     --SESSION 1 --N_EPOCHS 12 --BATCH_SIZE 6 --LEARNING_RATE 3e-5 \
#     --TRAINING 1 --ROUTING confidence \
#     --TEACHER_EN_CKPT "$TEACHER_EN_CKPT" \
#     --TEACHER_FR_CKPT "$TEACHER_FR_CKPT" \
#     --INCLUDE_FI 0 --SAVE_DIR /content/checkpoints \
#     --HF_DATASET_REPO "$HF_DATASET_REPO"


## 7. Model Evaluation & Confusion Matrix
Evaluate any saved checkpoint on the test set by setting `--TRAINING 0`.

In [ ]:
!python main.py --PARADIGM MTKD --LANGUAGE EN --LINGUALITY Multilingual \
    --SESSION 1 --TRAINING 0 --ROUTING language \
    --TEACHER_EN_CKPT "$TEACHER_EN_CKPT" \
    --TEACHER_FR_CKPT "$TEACHER_FR_CKPT" \
    --INCLUDE_FI 0 --SAVE_DIR /content/checkpoints \
    --HF_DATASET_REPO "$HF_DATASET_REPO" \
    --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1


## 8. Monitor Training (Weights & Biases & TensorBoard)

- **Weights & Biases Dashboard**: Check your live training runs, confusion matrices, and model artifacts at [wandb.ai](https://wandb.ai).
- **TensorBoard**: Optionally visualize local logs via the embedded TensorBoard widget below.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/checkpoints/tb